# Regression Data Visualization and Analysis

This notebook provides comprehensive visualizations for the regression dataset to understand data patterns, distributions, temporal trends, and relationships between features.

**Dataset:** Combined commercial banks stock price data

**Purpose:** Exploratory data analysis for regression modeling

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully")

## 2. Load Data

In [ ]:
# Load combined dataset
data = pd.read_csv('../data_preprocessing/combined_banks_dataset.csv')
data['published_date'] = pd.to_datetime(data['published_date'])
data = data.sort_values(['company_id', 'published_date']).reset_index(drop=True)

print(f"Dataset shape: {data.shape}")
print(f"Number of banks: {data['company_id'].nunique()}")
print(f"Date range: {data['published_date'].min()} to {data['published_date'].max()}")
print(f"\nColumns: {list(data.columns)}")

In [ ]:
# Display first few rows
data.head(10)

In [ ]:
# Dataset information
data.info()

In [ ]:
# Statistical summary
data.describe()

## 3. Missing Values Analysis

In [ ]:
# Check for missing values
missing = data.isnull().sum()
missing_pct = (missing / len(data)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})

print("Missing Values:")
print(missing_df[missing_df['Missing Count'] > 0])

if missing.sum() == 0:
    print("\nNo missing values found")

## 4. Data Distribution per Bank

In [ ]:
# Sample count per bank
bank_counts = data['company_id'].value_counts().sort_values(ascending=False)

plt.figure(figsize=(14, 6))
bank_counts.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Number of Samples per Bank', fontsize=14, fontweight='bold')
plt.xlabel('Bank', fontsize=12)
plt.ylabel('Number of Samples', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nTotal samples: {len(data):,}")
print(f"Average samples per bank: {bank_counts.mean():.0f}")
print(f"Median samples per bank: {bank_counts.median():.0f}")
print(f"Min samples: {bank_counts.min()}")
print(f"Max samples: {bank_counts.max()}")

## 5. Price Distribution Analysis

In [ ]:
# Distribution of closing prices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(data['close'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Closing Prices', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Closing Price', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].grid(alpha=0.3)

# Box plot
axes[1].boxplot(data['close'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Box Plot of Closing Prices', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Closing Price', fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean closing price: {data['close'].mean():.2f}")
print(f"Median closing price: {data['close'].median():.2f}")
print(f"Std closing price: {data['close'].std():.2f}")

## 6. Price Range by Bank

In [ ]:
# Average closing price per bank
avg_price_by_bank = data.groupby('company_id')['close'].mean().sort_values(ascending=False)

plt.figure(figsize=(14, 6))
avg_price_by_bank.plot(kind='bar', color='coral', edgecolor='black')
plt.title('Average Closing Price by Bank', fontsize=14, fontweight='bold')
plt.xlabel('Bank', fontsize=12)
plt.ylabel('Average Closing Price', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Price range (min, max) per bank
price_range = data.groupby('company_id')['close'].agg(['min', 'max', 'mean'])
price_range['range'] = price_range['max'] - price_range['min']
price_range = price_range.sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(price_range))
width = 0.6

ax.bar(x, price_range['range'], width, label='Price Range', color='lightcoral', edgecolor='black')
ax.plot(x, price_range['mean'], 'o-', color='darkblue', linewidth=2, markersize=8, label='Average Price')

ax.set_title('Price Range and Average by Bank', fontsize=14, fontweight='bold')
ax.set_xlabel('Bank', fontsize=12)
ax.set_ylabel('Price', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(price_range.index, rotation=45, ha='right')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Temporal Trends

In [ ]:
# Overall price trend over time
daily_avg = data.groupby('published_date')['close'].mean()

plt.figure(figsize=(14, 6))
plt.plot(daily_avg.index, daily_avg.values, linewidth=1.5, color='steelblue')
plt.title('Average Closing Price Over Time (All Banks)', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Average Closing Price', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Price trend for top 5 banks by average price
top_5_banks = avg_price_by_bank.head(5).index

plt.figure(figsize=(14, 6))

for bank in top_5_banks:
    bank_data = data[data['company_id'] == bank]
    plt.plot(bank_data['published_date'], bank_data['close'], label=bank, linewidth=1.5, alpha=0.8)

plt.title('Closing Price Trends - Top 5 Banks by Average Price', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Closing Price', fontsize=12)
plt.legend(loc='best', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Volume Analysis

In [ ]:
# Trading volume distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(data['traded_quantity'], bins=50, color='orange', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Trading Volume', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Traded Quantity', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].grid(alpha=0.3)

# Log scale histogram for better visualization
axes[1].hist(np.log1p(data['traded_quantity']), bins=50, color='green', edgecolor='black', alpha=0.7)
axes[1].set_title('Distribution of Trading Volume (Log Scale)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Log(Traded Quantity + 1)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Average trading volume per bank
avg_volume_by_bank = data.groupby('company_id')['traded_quantity'].mean().sort_values(ascending=False)

plt.figure(figsize=(14, 6))
avg_volume_by_bank.plot(kind='bar', color='teal', edgecolor='black')
plt.title('Average Trading Volume by Bank', fontsize=14, fontweight='bold')
plt.xlabel('Bank', fontsize=12)
plt.ylabel('Average Trading Volume', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Price-Volume Relationship

In [ ]:
# Scatter plot: Price vs Volume
plt.figure(figsize=(12, 6))
plt.scatter(data['traded_quantity'], data['close'], alpha=0.3, s=10, color='purple')
plt.title('Closing Price vs Trading Volume', fontsize=14, fontweight='bold')
plt.xlabel('Traded Quantity', fontsize=12)
plt.ylabel('Closing Price', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate correlation
correlation = data['close'].corr(data['traded_quantity'])
print(f"Correlation between closing price and trading volume: {correlation:.4f}")

## 10. Daily Price Change Analysis

In [ ]:
# Distribution of percentage change
plt.figure(figsize=(14, 6))
plt.hist(data['per_change'].dropna(), bins=100, color='crimson', edgecolor='black', alpha=0.7)
plt.axvline(x=0, color='black', linestyle='--', linewidth=2, label='Zero Change')
plt.title('Distribution of Daily Percentage Change', fontsize=14, fontweight='bold')
plt.xlabel('Percentage Change (%)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean percentage change: {data['per_change'].mean():.4f}%")
print(f"Std percentage change: {data['per_change'].std():.4f}%")
print(f"Positive changes: {(data['per_change'] > 0).sum()} ({(data['per_change'] > 0).sum() / len(data) * 100:.2f}%)")
print(f"Negative changes: {(data['per_change'] < 0).sum()} ({(data['per_change'] < 0).sum() / len(data) * 100:.2f}%)")

## 11. OHLC Analysis

In [ ]:
# Open, High, Low, Close comparison
ohlc_cols = ['open', 'high', 'low', 'close']
ohlc_data = data[ohlc_cols]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['steelblue', 'green', 'red', 'orange']

for idx, (col, ax, color) in enumerate(zip(ohlc_cols, axes.flatten(), colors)):
    ax.hist(data[col], bins=50, color=color, edgecolor='black', alpha=0.7)
    ax.set_title(f'Distribution of {col.upper()} Price', fontsize=12, fontweight='bold')
    ax.set_xlabel('Price', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Daily price range (high - low)
data['daily_range'] = data['high'] - data['low']

plt.figure(figsize=(14, 6))
plt.hist(data['daily_range'], bins=50, color='purple', edgecolor='black', alpha=0.7)
plt.title('Distribution of Daily Price Range (High - Low)', fontsize=14, fontweight='bold')
plt.xlabel('Price Range', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Average daily range: {data['daily_range'].mean():.2f}")
print(f"Max daily range: {data['daily_range'].max():.2f}")

## 12. Correlation Analysis

In [ ]:
# Correlation matrix for numerical features
numeric_cols = ['open', 'high', 'low', 'close', 'traded_quantity', 'total_traded_value', 'per_change']
correlation_matrix = data[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.3f', cmap='coolwarm', 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 13. Volatility Analysis

In [ ]:
# Calculate volatility (rolling standard deviation of percentage change) per bank
volatility_by_bank = data.groupby('company_id')['per_change'].std().sort_values(ascending=False)

plt.figure(figsize=(14, 6))
volatility_by_bank.plot(kind='bar', color='darkred', edgecolor='black')
plt.title('Volatility by Bank (Std of Percentage Change)', fontsize=14, fontweight='bold')
plt.xlabel('Bank', fontsize=12)
plt.ylabel('Volatility (Standard Deviation)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Most volatile bank: {volatility_by_bank.index[0]} (Std: {volatility_by_bank.iloc[0]:.4f})")
print(f"Least volatile bank: {volatility_by_bank.index[-1]} (Std: {volatility_by_bank.iloc[-1]:.4f})")

## 14. Monthly Trends

In [ ]:
# Extract month and year
data['year'] = data['published_date'].dt.year
data['month'] = data['published_date'].dt.month
data['year_month'] = data['published_date'].dt.to_period('M')

# Monthly average price
monthly_avg = data.groupby('year_month')['close'].mean()

plt.figure(figsize=(14, 6))
monthly_avg.plot(kind='line', linewidth=2, color='navy', marker='o', markersize=4)
plt.title('Monthly Average Closing Price', fontsize=14, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Average Closing Price', fontsize=12)
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 15. Summary Statistics by Bank

In [ ]:
# Summary statistics for each bank
summary_by_bank = data.groupby('company_id').agg({
    'close': ['mean', 'std', 'min', 'max'],
    'traded_quantity': 'mean',
    'per_change': ['mean', 'std']
}).round(2)

summary_by_bank.columns = ['_'.join(col).strip() for col in summary_by_bank.columns.values]
summary_by_bank = summary_by_bank.sort_values('close_mean', ascending=False)

print("Summary Statistics by Bank:")
print("="*80)
display(summary_by_bank)

## 16. Data Quality Check

In [ ]:
# Check for outliers using IQR method
Q1 = data['close'].quantile(0.25)
Q3 = data['close'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = data[(data['close'] < lower_bound) | (data['close'] > upper_bound)]

print(f"Number of outliers in closing price: {len(outliers)}")
print(f"Percentage of outliers: {len(outliers) / len(data) * 100:.2f}%")
print(f"\nIQR bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")

In [ ]:
# Check for duplicate dates per company
duplicates = data.groupby(['company_id', 'published_date']).size()
duplicates = duplicates[duplicates > 1]

if len(duplicates) > 0:
    print(f"Number of duplicate date entries: {len(duplicates)}")
    print(duplicates.head())
else:
    print("No duplicate date entries found per company")

## 17. Temporal Coverage by Bank

In [ ]:
# Date range coverage per bank
date_coverage = data.groupby('company_id')['published_date'].agg(['min', 'max', 'count'])
date_coverage['days_span'] = (date_coverage['max'] - date_coverage['min']).dt.days
date_coverage = date_coverage.sort_values('days_span', ascending=False)

print("Temporal Coverage by Bank:")
print("="*80)
display(date_coverage)

In [ ]:
# Visualize temporal coverage
fig, ax = plt.subplots(figsize=(14, 8))

for idx, bank in enumerate(date_coverage.index):
    bank_data = date_coverage.loc[bank]
    ax.barh(idx, bank_data['days_span'], left=bank_data['min'], 
            height=0.8, color='skyblue', edgecolor='black')

ax.set_yticks(range(len(date_coverage)))
ax.set_yticklabels(date_coverage.index)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Bank', fontsize=12)
ax.set_title('Temporal Coverage by Bank', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 18. Conclusion

This visualization analysis provides comprehensive insights into the regression dataset:

1. Data distribution across banks is relatively balanced
2. Price ranges vary significantly between different banks
3. Temporal trends show various patterns in stock prices
4. Strong correlation between OHLC prices as expected
5. Trading volume varies considerably across banks
6. Daily percentage changes follow a near-normal distribution
7. Volatility differs between banks, indicating varying risk levels

These visualizations support the modeling approach of using simple features (close price and momentum) for regression prediction.